# 01 SFT + LoRA 最小训练闭环：从一条数据到 adapter

你现在已经读完 `scripts/infer.py`。下一步不要看 DPO，不要看 reports，不要看一堆历史实验。

今天只看一个核心文件：

```text
scripts/train_sft_lora.py
```

目标不是“把代码看一遍”，而是建立代码地图：

```text
一条 SFT JSONL
  -> ChatSFTDataset 读取 messages
  -> encode() 用 tokenizer 变成 input_ids / labels
  -> labels 里 prompt 部分是 -100，只学 assistant
  -> load_model() 加载 Qwen base
  -> LoraConfig + get_peft_model 挂 LoRA
  -> Trainer.train() 训练
  -> trainer.save_model() 保存 adapter
```

In [1]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/codex备份/qwen-local-lora-sft-dpo学习版")

print("项目根目录:", PROJECT_ROOT)

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=80):
    lines = read_text(rel).splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

项目根目录: D:\coding\codex备份\qwen-local-lora-sft-dpo学习版


## 0. 今天只看哪些真实项目文件？

- `\scripts\infer.py`是加载lora adapter做推理。（如果有lora adapter，没有的话）
- `scripts/train_sft_lora.py`是训练lora adapter。
- `data/processed/custom_sft_train.jsonl` 是真实的lora训练数据（我们的sft就是用lora来实现的）
 
| 类型 | 真实路径 | 今天看它干嘛 |
|---|---|---|
| 训练主入口 | `scripts/train_sft_lora.py` | SFT + LoRA 的全部核心逻辑都在这里 |
| SFT 样本 | `data/processed/custom_sft_train.jsonl` | 看一条真实训练数据长什么样 |
| 推理入口 | `scripts/infer.py` | 训练后用它加载 adapter 验证回答 |

先别看别的。你只要能把这 3 个位置连起来，就抓住了项目第二层楼。

In [2]:
for rel in ["scripts/train_sft_lora.py", "data/processed/custom_sft_train.jsonl", "scripts/infer.py"]:
    p = PROJECT_ROOT / rel
    print(f"{rel:45s}", "OK" if p.exists() else "MISSING")

scripts/train_sft_lora.py                     OK
data/processed/custom_sft_train.jsonl         OK
scripts/infer.py                              OK


## 1. 先看一条真实 SFT 数据

入口：`data/processed/custom_sft_train.jsonl`

SFT 数据不是神秘的东西。每一行就是一个 JSON，每个 JSON 里有 `messages`。

In [ ]:
print("custom_sft_train 行数:", count_jsonl("data/processed/custom_sft_train.jsonl"))
row = read_jsonl("data/processed/custom_sft_train.jsonl", n=1)[0]
print(json.dumps(row, ensure_ascii=False, indent=2)[:2500])

你要先记住这个形状：

```text
messages = [
  {role: system, content: 规则/身份},
  {role: user, content: 用户问题},
  {role: assistant, content: 标准答案}
]
```

SFT 的本质就是：让模型看到 system + user 后，学习 assistant 这段标准答案。

## 2. train_sft_lora.py 的核心代码地图

不要从第一行读到最后一行。先按入口函数地图读：

| 代码位置 | 你要看什么 | 输入 | 输出 |
|---|---|---|---|
| `ChatSFTDataset.__init__` | 读 JSONL | 文件路径 | examples 列表 |
| `ChatSFTDataset.encode` | 把 messages 变 token/labels | messages | ChatExample |
| `DataCollatorForChatSFT` | padding 成 batch | 多个 ChatExample | tensor batch |
| `load_tokenizer` | 加载 tokenizer | model_name | tokenizer |
| `load_model` | 加载 Qwen 并挂 LoRA | args | PEFT model |
| `main` | 串起训练 | 命令行参数 | adapter 输出目录 |

In [1]:
for keyword in [
    "class ChatSFTDataset",
    "def encode",
    "class DataCollatorForChatSFT",
    "def load_tokenizer",
    "def load_model",
    "def main",
]:
    find_lines("scripts/train_sft_lora.py", keyword, context=3)

NameError: name 'find_lines' is not defined

## 3. 数据怎么读进来？看 `ChatSFTDataset.__init__`

真实代码位置：`scripts/train_sft_lora.py` 里的 `class ChatSFTDataset`。

它做的事：

```text
打开 train_file
  -> 一行一行读 JSON
  -> 取 row["messages"]
  -> self.encode(messages)
  -> 存进 self.examples
```

In [2]:
find_lines("scripts/train_sft_lora.py", "with Path(path).open", context=10)
find_lines("scripts/train_sft_lora.py", "self.examples.append", context=6)

NameError: name 'find_lines' is not defined

## 4. 最核心：SFT 到底学哪一段？看 `encode()`

这是你必须吃透的核心。

真实代码位置：`scripts/train_sft_lora.py -> ChatSFTDataset.encode`。

In [ ]:
find_lines("scripts/train_sft_lora.py", "prompt_messages = messages[:-1]", context=16)
find_lines("scripts/train_sft_lora.py", "labels = [IGNORE_INDEX]", context=8)

这段代码的意义：

```text
prompt_messages = system + user
assistant_message = 最后一条 assistant 标准答案

prompt_ids = tokenizer(system + user + “轮到 assistant 回答”)
full_ids = tokenizer(system + user + assistant 标准答案)

labels = [-100, -100, ..., assistant token ids]
```

`-100` 的意思是：这部分不算 loss。

所以模型不是在学习背诵 system/user，模型只因为 assistant 标准答案预测得不好而扣分。

## 5. LoRA 在哪里挂上去？看 `load_model()`

这里是项目从普通 Qwen 变成 LoRA 训练的关键。

In [ ]:
find_lines("scripts/train_sft_lora.py", "AutoModelForCausalLM.from_pretrained", context=12)
find_lines("scripts/train_sft_lora.py", "LoraConfig", context=16)
find_lines("scripts/train_sft_lora.py", "get_peft_model", context=6)

你要能讲清楚：

- `AutoModelForCausalLM.from_pretrained`：加载 Qwen base。
- `LoraConfig`：定义 LoRA 小补丁的大小、强度、贴到哪些层。
- `target_modules`：LoRA 贴到 `q_proj/k_proj/v_proj/o_proj/gate_proj/up_proj/down_proj`。
- `get_peft_model`：真正把 LoRA adapter 挂到 Qwen 上。

所以 LoRA 不是另一种数据格式，而是一种“训练哪些参数”的方法。

## 6. 训练入口在哪里？看 `Trainer` 和 `trainer.train()`

In [ ]:
find_lines("scripts/train_sft_lora.py", "TrainingArguments", context=16)
find_lines("scripts/train_sft_lora.py", "trainer = Trainer", context=14)
find_lines("scripts/train_sft_lora.py", "trainer.train", context=8)
find_lines("scripts/train_sft_lora.py", "trainer.save_model", context=6)

这里就是完整出口：

```text
Trainer.train()
  -> 更新 LoRA 参数
  -> trainer.save_model(output_dir)
  -> tokenizer.save_pretrained(output_dir)
  -> outputs/xxx 里出现 adapter
```

你要背成一句话：

> `train_sft_lora.py` 的入口是 SFT JSONL，核心是 `ChatSFTDataset.encode` 把 messages 变成 input_ids 和 labels，其中 prompt 部分 labels 是 -100；然后 `load_model` 加载 Qwen base 并通过 `LoraConfig + get_peft_model` 挂 LoRA；最后 `Trainer.train()` 只训练 LoRA adapter，并保存到 outputs。